In [1]:
import glob
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
import lightgbm as lgb

# 1. Load competition files
input_files = glob.glob("/kaggle/input/**/*", recursive=True)
train_path = next((p for p in input_files if "train" in os.path.basename(p)), "train.csv.gz")
test_path  = next((p for p in input_files if "test" in os.path.basename(p)), "test.csv")
sub_path   = next((p for p in input_files if "sample_submission" in os.path.basename(p)), "sample_submission.csv")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)
sub_df   = pd.read_csv(sub_path)

TOP_FACILITIES = [
    "FCL-007", "FCL-012", "FCL-035", "FCL-055", "FCL-056", "FCL-058",
    "FCL-072", "FCL-077", "FCL-078", "FCL-080", "FCL-085", "FCL-087",
    "FCL-093", "FCL-104", "FCL-135", "FCL-136", "FCL-137", "FCL-138", "FCL-139", "FCL-141"
]

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.drop(columns=["price_tier"], errors="ignore").copy()

    # Facilities count & presence flags
    if "facilities" in out.columns:
        s = out["facilities"].fillna("").astype(str)
        out["facility_count"] = np.where(s.str.len() > 2, s.str.count(",") + 1, 0).astype("float32")
        for code in TOP_FACILITIES:
            out[f"has_{code}"] = s.str.contains(code, regex=False).astype(np.int8)

    # Date & duration features
    for dcol in ["snapshot_start_date", "snapshot_end_date", "first_review_date", "last_review_date"]:
        if dcol in out.columns:
            dt = pd.to_datetime(out[dcol], errors="coerce")
            out[f"{dcol}_yr"] = dt.dt.year.astype("float32")
            out[f"{dcol}_mo"] = dt.dt.month.astype("float32")
            out[f"{dcol}_ord"] = np.where(
                dt.notna(), dt.values.astype("datetime64[D]").astype("float64"), np.nan
            ).astype("float32")

    if {"snapshot_start_date_ord", "snapshot_end_date_ord"}.issubset(out.columns):
        out["snapshot_span_days"] = (out["snapshot_end_date_ord"] - out["snapshot_start_date_ord"]).astype("float32")
    if {"first_review_date_ord", "last_review_date_ord"}.issubset(out.columns):
        out["review_span_days"] = (out["last_review_date_ord"] - out["first_review_date_ord"]).astype("float32")
    if {"snapshot_end_date_ord", "last_review_date_ord"}.issubset(out.columns):
        out["days_since_last_review"] = (out["snapshot_end_date_ord"] - out["last_review_date_ord"]).astype("float32")

    # Missingness & tenure
    if "review_count" in out.columns:
        out["has_reviews"] = (out["review_count"] > 0).astype(np.int8)
    if "borough" in out.columns:
        out["has_borough"] = out["borough"].notna().astype(np.int8)
    out["missing_cnt"] = out.isna().sum(axis=1).astype("float32")

    if {"operator_platform_age_years", "operator_platform_age_months"}.issubset(out.columns):
        out["op_platform_months"] = (
            out["operator_platform_age_years"].fillna(0) * 12 + out["operator_platform_age_months"].fillna(0)
        ).astype("float32")
    if {"operator_active_years", "operator_active_months"}.issubset(out.columns):
        out["op_active_months"] = (
            out["operator_active_years"].fillna(0) * 12 + out["operator_active_months"].fillna(0)
        ).astype("float32")

    # Capacity, utilization & availability ratios
    if {"max_seats", "workstations"}.issubset(out.columns):
        out["seats_per_ws"] = (out["max_seats"] / (out["workstations"].fillna(0) + 1.0)).astype("float32")
    if {"max_seats", "zones"}.issubset(out.columns):
        out["seats_per_zone"] = (out["max_seats"] / (out["zones"].fillna(0) + 1.0)).astype("float32")
    if {"utility_rooms", "max_seats"}.issubset(out.columns):
        out["util_per_seat"] = (out["utility_rooms"].fillna(0) / (out["max_seats"] + 1.0)).astype("float32")
    if {"facility_count", "max_seats"}.issubset(out.columns):
        out["fac_per_seat"] = (out["facility_count"] / (out["max_seats"] + 1.0)).astype("float32")
    if {"operator_metro_units", "operator_portfolio_size"}.issubset(out.columns):
        out["metro_share"] = (out["operator_metro_units"] / (out["operator_portfolio_size"].fillna(0) + 1.0)).astype("float32")
    if {"booked_load", "max_seats"}.issubset(out.columns):
        out["load_per_seat"] = (out["booked_load"] / (out["max_seats"] + 1.0)).astype("float32")
    if {"review_count_12m", "review_count"}.issubset(out.columns):
        out["recent_12m_ratio"] = (out["review_count_12m"] / (out["review_count"] + 1.0)).astype("float32")
    if {"open_slots_a", "open_slots_d", "open_slots_e"}.issubset(out.columns):
        out["slot_sum_ade"] = (out["open_slots_a"] + out["open_slots_d"] + out["open_slots_e"]).astype("float32")

    score_cols = [
        c for c in ["score_overall", "score_accuracy", "score_condition",
                    "score_onboarding", "score_support", "score_access", "score_value"]
        if c in out.columns
    ]
    if score_cols:
        out["score_mean_7"] = out[score_cols].mean(axis=1).astype("float32")
        out["score_std_7"] = out[score_cols].std(axis=1).astype("float32")

    # Clip extreme tail values
    for col, cap in [("max_term_days", 3650.0), ("max_term_floor", 3650.0),
                      ("max_term_ceiling", 3650.0), ("max_term_avg", 3650.0),
                      ("reviews_per_month", 30.0)]:
        if col in out.columns:
            out[col] = np.clip(out[col].astype("float64"), 0.0, cap).astype("float32")

    return out.drop(columns=["facilities", "snapshot_start_date", "snapshot_end_date",
                            "first_review_date", "last_review_date"], errors="ignore")

y = train_df["price_tier"].astype(np.int8)
X_tr = build_features(train_df)
X_te = build_features(test_df)

# Unlabelled frequency & group aggregations across combined train + test
both = pd.concat([X_tr, X_te], axis=0, ignore_index=True)
for gcol in ["operator_ref", "zone", "space_class", "metro", "borough"]:
    if gcol in both.columns:
        freq = both[gcol].value_counts(normalize=True)
        X_tr[f"{gcol}_freq"] = X_tr[gcol].map(freq).fillna(0.0).astype("float32")
        X_te[f"{gcol}_freq"] = X_te[gcol].map(freq).fillna(0.0).astype("float32")

for grp, num_f in [("metro", "max_seats"), ("metro", "facility_count"), ("access_type", "max_seats")]:
    if {grp, num_f}.issubset(both.columns):
        lookup = both.groupby(grp, dropna=False)[num_f].mean().to_dict()
        X_tr[f"{num_f}_mean_{grp}"] = X_tr[grp].map(lookup).astype("float32")
        X_te[f"{num_f}_mean_{grp}"] = X_te[grp].map(lookup).astype("float32")
        X_tr[f"{num_f}_diff_{grp}"] = (X_tr[num_f] - X_tr[f"{num_f}_mean_{grp}"]).astype("float32")
        X_te[f"{num_f}_diff_{grp}"] = (X_te[num_f] - X_te[f"{num_f}_mean_{grp}"]).astype("float32")

# Out-of-fold smoothed multi-class target encoding (computed strictly inside CV)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
te_cols = [c for c in ["operator_ref", "zone", "space_class", "borough"] if c in X_tr.columns]
prior = np.bincount(y)[1:] / len(y)
smoothing = 20.0

oof_te = np.zeros((len(X_tr), len(te_cols) * 4), dtype=np.float32)
test_te = np.zeros((len(X_te), len(te_cols) * 4), dtype=np.float32)

for tr_idx, va_idx in skf.split(X_tr, y):
    y_oh = np.eye(4)[y.iloc[tr_idx].values - 1]
    off = 0
    for col in te_cols:
        s_tr = X_tr.iloc[tr_idx][col].fillna("MISSING").astype(str)
        s_va = X_tr.iloc[va_idx][col].fillna("MISSING").astype(str)
        tmp = pd.DataFrame(y_oh, columns=[f"c{k}" for k in range(4)])
        tmp["k"] = s_tr.values
        probs = (tmp.groupby("k").sum().values + smoothing * prior[None, :]) / (tmp.groupby("k").size().values[:, None] + smoothing)
        pdf = pd.DataFrame(probs, index=tmp.groupby("k").sum().index, columns=[f"c{k}" for k in range(4)])
        oof_te[np.ix_(va_idx, range(off, off + 4))] = pdf.reindex(s_va).fillna(pd.Series(prior, index=pdf.columns)).values.astype("float32")
        off += 4

y_oh_full = np.eye(4)[y.values - 1]
off = 0
for col in te_cols:
    s_tr = X_tr[col].fillna("MISSING").astype(str)
    s_te = X_te[col].fillna("MISSING").astype(str)
    tmp = pd.DataFrame(y_oh_full, columns=[f"c{k}" for k in range(4)])
    tmp["k"] = s_tr.values
    probs = (tmp.groupby("k").sum().values + smoothing * prior[None, :]) / (tmp.groupby("k").size().values[:, None] + smoothing)
    pdf = pd.DataFrame(probs, index=tmp.groupby("k").sum().index, columns=[f"c{k}" for k in range(4)])
    test_te[:, off:off + 4] = pdf.reindex(s_te).fillna(pd.Series(prior, index=pdf.columns)).values.astype("float32")
    off += 4

te_names = [f"te_{c}_t{t}" for c in te_cols for t in range(1, 5)]
X_tr = pd.concat([X_tr.reset_index(drop=True), pd.DataFrame(oof_te, columns=te_names)], axis=1)
X_te = pd.concat([X_te.reset_index(drop=True), pd.DataFrame(test_te, columns=te_names)], axis=1)

# Encode object columns
for c in X_tr.select_dtypes(include="object").columns:
    vals = pd.concat([X_tr[c], X_te[c]], axis=0).fillna("MISSING").astype(str)
    mapping = {v: i for i, v in enumerate(vals.unique())}
    X_tr[c] = X_tr[c].fillna("MISSING").astype(str).map(mapping).astype("int32")
    X_te[c] = X_te[c].fillna("MISSING").astype(str).map(mapping).astype("int32")

# Train & blend HistGradientBoosting + LightGBM across CV folds
oof_hgb = np.zeros((len(X_tr), 4), dtype=np.float64)
oof_lgb = np.zeros((len(X_tr), 4), dtype=np.float64)
test_hgb = np.zeros((len(X_te), 4), dtype=np.float64)
test_lgb = np.zeros((len(X_te), 4), dtype=np.float64)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr, y)):
    hgb = HistGradientBoostingClassifier(
        learning_rate=0.07, max_iter=300, max_leaf_nodes=95,
        min_samples_leaf=25, l2_regularization=1.5, random_state=42 + fold
    )
    hgb.fit(X_tr.iloc[tr_idx], y.iloc[tr_idx])
    oof_hgb[va_idx] = hgb.predict_proba(X_tr.iloc[va_idx])
    test_hgb += hgb.predict_proba(X_te) / 3.0

    gbm = lgb.LGBMClassifier(
        n_estimators=350, learning_rate=0.06, num_leaves=95,
        min_child_samples=30, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8, reg_lambda=1.5, random_state=42 + fold,
        n_jobs=-1, verbose=-1
    )
    gbm.fit(X_tr.iloc[tr_idx], y.iloc[tr_idx] - 1)
    oof_lgb[va_idx] = gbm.predict_proba(X_tr.iloc[va_idx])
    test_lgb += gbm.predict_proba(X_te) / 3.0

print(f"OOF Acc - HGB: {accuracy_score(y, oof_hgb.argmax(axis=1) + 1):.5f}")
print(f"OOF Acc - LGB: {accuracy_score(y, oof_lgb.argmax(axis=1) + 1):.5f}")

best_w, best_acc = 0.5, 0.0
for w in np.linspace(0.0, 1.0, 11):
    acc = accuracy_score(y, (w * oof_hgb + (1 - w) * oof_lgb).argmax(axis=1) + 1)
    if acc > best_acc:
        best_acc, best_w = acc, w

print(f"OOF Acc - Blend (w_hgb={best_w:.1f}): {best_acc:.5f}")

final_test_proba = best_w * test_hgb + (1 - best_w) * test_lgb
final_preds = final_test_proba.argmax(axis=1) + 1

sub = pd.DataFrame({
    "ID": sub_df["ID"],
    "price_tier": final_preds.astype(int),
})
sub.to_csv("submission.csv", index=False)
print("Saved submission.csv | shape:", sub.shape)

OOF Acc - HGB: 0.73138
OOF Acc - LGB: 0.73312
OOF Acc - Blend (w_hgb=0.5): 0.73511
Saved submission.csv | shape: (114538, 2)
